# GradientQL: scan DVGA in Colab

This notebook runs [**GradientQL**](https://github.com/ccc909/GradientQL), an autonomous GraphQL
vulnerability scanner driven by a single language model, against a local copy of the
[Damn Vulnerable GraphQL Application (DVGA)](https://github.com/dolevf/Damn-Vulnerable-GraphQL-Application):
an intentionally vulnerable target that is safe to attack.

You need an [OpenRouter](https://openrouter.ai) API key. The scan makes model calls that cost money
(a short run is a few cents). Run the cells top to bottom.

> ⚠️ **Only ever point GradientQL at a target you own or are authorized to test.** This notebook
> scans the bundled DVGA on localhost and nothing else.


In [ ]:
#@title 1. Install GradientQL
!pip install -q "git+https://github.com/ccc909/GradientQL.git"
print("GradientQL installed.")


In [ ]:
#@title 2. Start DVGA (the target) in the background  [first run takes a few minutes]
import os
import pathlib
import shutil
import subprocess
import time
import urllib.request

DVGA = "/content/dvga"
VENV = "/content/dvga-venv"
URL = "http://127.0.0.1:5013/graphql"
LOG = "/content/dvga_server.log"
py = f"{VENV}/bin/python"

# Treat the venv as ready only if its python can import gevent. A partial venv left by an earlier
# failed run (Colab's 3.12 venv that could not bootstrap pip) would otherwise be reused and break.
ready = os.path.isfile(py) and subprocess.run(
    [py, "-c", "import gevent"], capture_output=True).returncode == 0

# DVGA targets an older Python (Flask 2.2 / graphene 2.x) that breaks on Colab's Python 3.12, so give
# it its own Python 3.9 venv, isolated from GradientQL. The first build takes a couple of minutes; the
# steps below print as they go so you can see progress.
if not ready:
    shutil.rmtree(VENV, ignore_errors=True)
    print("[1/4] installing Python 3.9 (deadsnakes) ...", flush=True)
    for cmd in (
        "apt-get install -y -qq software-properties-common",
        "add-apt-repository -y ppa:deadsnakes/ppa",
        "apt-get update -qq",
        "apt-get install -y -qq python3.9 python3.9-venv",
    ):
        subprocess.run(cmd, shell=True, check=True)
    if not os.path.isdir(DVGA):
        print("[2/4] cloning DVGA ...", flush=True)
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/dolevf/Damn-Vulnerable-GraphQL-Application.git", DVGA], check=True)
    print("[3/4] building the DVGA venv and installing its deps (~1-2 min) ...", flush=True)
    subprocess.run(["python3.9", "-m", "venv", VENV], check=True)
    subprocess.run([py, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
    subprocess.run([py, "-m", "pip", "install", "-q", "-r", f"{DVGA}/requirements.txt"], check=True)

# The stock DVGA never calls monkey.patch_all(), so a single blocking resolver freezes the whole
# server and a scan wedges on every step. Prepend it (idempotent) so concurrent requests yield.
app = pathlib.Path(f"{DVGA}/app.py")
src = app.read_text()
if "monkey.patch_all" not in src:
    app.write_text("from gevent import monkey; monkey.patch_all()\n" + src)

print("[4/4] starting DVGA ...", flush=True)
env = {**os.environ, "WEB_HOST": "127.0.0.1", "WEB_PORT": "5013"}
logf = open(LOG, "wb")
subprocess.Popen([py, "app.py"], cwd=DVGA, env=env, stdout=logf, stderr=subprocess.STDOUT)

probe = urllib.request.Request(URL, data=b'{"query":"{__typename}"}',
                               headers={"Content-Type": "application/json"})
for _ in range(60):
    try:
        if urllib.request.urlopen(probe, timeout=3).status == 200:
            print("DVGA is up at", URL)
            break
    except Exception:
        time.sleep(2)
else:
    print("DVGA did not come up. Last lines of its server log:\n")
    print(pathlib.Path(LOG).read_text()[-3000:])
    raise RuntimeError("DVGA did not start (see the log above).")


In [ ]:
#@title 3. Enter your OpenRouter API key
import getpass
import os
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key (input hidden): ").strip()
print("Key set." if os.environ["OPENROUTER_API_KEY"] else "No key entered - the scan will not run.")


In [ ]:
#@title 4. Run the scan
MODEL = "z-ai/glm-5.2"  #@param ["z-ai/glm-5.2", "openai/gpt-oss-120b", "qwen/qwen3.7-max"]
BUDGET = 20  #@param {type:"slider", min:5, max:60, step:5}

with open("demo.yaml", "w") as f:
    f.write(f"scanner:\n  budget: {BUDGET}\nllm:\n  attacker_model: \"{MODEL}\"\n")

# --no-tui prints plain logs (Colab has no interactive terminal); --trace writes a full step log.
!python -m gradientql --settings demo.yaml --url http://127.0.0.1:5013/graphql --no-tui --trace


## What just happened

GradientQL introspected DVGA, worked through the attack surface on its own, and printed a report of
what it confirmed and retracted. Because `--trace` was on, a full step-by-step log was written under
`output/` (open the file browser on the left of Colab); the findings alone are in
`output/vuln_stream.jsonl`.

- **Go deeper:** raise `BUDGET` and re-run the scan cell. glm finds the most; `gpt-oss-120b` is the
  cheapest and fastest.
- **Scan your own target:** replace the `--url` with an endpoint you are authorized to test, and skip
  the DVGA cell.
- **More options:** see the [README](https://github.com/ccc909/GradientQL) and
  [Configuration](https://github.com/ccc909/GradientQL/blob/main/docs/CONFIGURATION.md).
